In [51]:
# 환경설정
import os
import sys
from tqdm import tqdm

# duckdb 활용
import duckdb

# langchain 라이브러리 활용
from langchain_openai import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore

from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 1) VectorDB 내부 전략
from hashlib import md5
from tqdm import tqdm


# es strategy 전략 
from langchain_elasticsearch import DenseVectorStrategy #   query vector 전략
from langchain_elasticsearch import BM25Strategy # 키워드 기반 전략

from uuid import uuid4

import pandas as pd
import numpy as np

# chain
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableMap
from langchain_core.prompts import ChatPromptTemplate


# 환경변수 세팅
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
con = duckdb.connect('../DB/news_articles.db')
news_df  = con.execute('select * from articles').fetch_df()
con.close()

In [4]:
news_df

,header,summary,content,url,datetime,ticker
0,관세 협상 앞두고 이재용·김동관·최태원 만난 李…무슨 얘기 나눴나,None,이재명 대통령이 6월 13일 서울 용산 대통령실청사에서 열린 6경제단체와 기업인 간...,https://magazine.hankyung.com/business/article...,2025.07.25 08:04,삼성전자
1,"SKT, 위약금 16만명 이탈…이젠 고객 쟁탈전",유심해킹 사고 이후 총 80만명 이탈 \n면제 이후 이탈자 예상보다 적어 \n폴더블...,영상 모듈 닫기\n\n\n\n\n<앵커>SK텔레콤이 해킹 사고 이후 위약금을 면제해...,https://www.hankyung.com/article/2025071501305,2025.07.15 17:23,삼성전자
2,"미국, 중국산 흑연에 반덤핑 관세 예고…2차전지 관련주 급등",None,미국이 중국산 흑연에 고율의 반덤핑 관세를 예고하면서 국내 2차전지 업종 주가가 일...,https://www.hankyung.com/article/202507211741a,2025.07.22 11:00,삼성전자
3,"갤럭시 워치8 시리즈, 가장 얇은 디자인…'웨어러블 혁신'의 새 기준 제시",삼성전자 '갤럭시 언팩 2025'\n\n뛰어난 착용감과 강력한 기능\n차별화된 쿠션...,‘갤럭시 워치8 클래식 46㎜ 화이트’. 삼성전자 제공\n\n\n ...,https://www.hankyung.com/article/2025072235641,2025.07.22 16:39,삼성전자
4,9~10% 주식 관련 대출에서 3%대 대출로 교체,None,"전송종목 : 하이트진로, 유한양행, CJ대한통운, 두산, DL최근 주식 투자자들 사...",https://www.hankyung.com/article/202507185994a,2025.07.18 08:33,삼성전자
...,...,...,...,...,...,...
11050,"미 마이크론·웨스턴디지털, 키옥시아 지분 거래 타진",None,미국 반도체 기업인 마이크론(Micron)과 웨스턴 디지털(Western Digit...,https://www.hankyung.com/article/202104018917Y,2021.04.01 15:48,ADVANCED MICRO DEVICES
11051,"NH투자증권, ELS 6종 모집…최대 연 12% 수익 추구",None,NH투자증권은 주가연계파생결합증권(ELS) 6종을 내달 1일 오후 1시까지 모집한다...,https://www.hankyung.com/article/2021033080886,2021.03.30 12:54,ADVANCED MICRO DEVICES
11052,美증시 주요지수 일제히 하락…다우 0.48%↓,None,사진=게티이미지뱅크 \n\n 뉴욕 증시가 일부 기업들 실...,https://www.hankyung.com/article/2017102633887,2017.10.26 06:28,ADVANCED MICRO DEVICES
11053,"한국투자증권, 테슬라·AMD-엔비디아 기초 ELS 2종 공모",None,한국투자증권은 뱅키스 전용 주가연계증권(ELS) 2종을 각각 50억원 한도로 공모한...,https://www.hankyung.com/article/2022060294825,2022.06.02 13:59,ADVANCED MICRO DEVICES


In [18]:
news_df_filer = news_df.groupby(['header','content','datetime','url'],as_index=False).agg({'ticker' : lambda x: x})

In [19]:
news_df_filer

,header,content,datetime,url,ticker
0,"""'150만명 고용' 車산업 위기…신정부, 미래차 지원·노조법 개정 재검토해야""",사진=뉴스1\n\n “자동차산업의 위기가 곧 국가 제조업...,2025.06.24 10:43,https://www.hankyung.com/article/202506245634i,기아
1,"""'미지의 서울' 대박난 거 맞아요?""…속 타는 개미들 [종목+]","미지의서울 이재인./사진=tvN\n\n ""새 정부 출범 ...",2025.07.17 08:29,https://www.hankyung.com/article/2025071733966,NAVER
2,"""'차세대 배터리 기술' 중국에 완전히 밀렸다""…쏟아진 우려",사진=게티이미지뱅크 \n\n 중국 82개 vs 한국 1개...,2025.07.11 15:54,https://www.hankyung.com/article/202507112930i,"[에코프로, 에코프로비엠]"
3,"""1000만원이 6000만원 됐다""…불기둥에 개미들 '환호' [종목+]",어린이날 경기 고양시 일산서구 현대 모터스튜디오 고양에서 인기 애니메이션 '캐치! ...,2025.06.19 08:55,https://www.hankyung.com/article/2025061949066,기아
4,"""10년 맡겼더니 3배 벌었다""…40대에 대박난 '재테크' 뭐길래",사진=게티이미지뱅크 \n\n 10년간 퇴직연금을 3배 가...,2025.07.19 07:25,https://www.hankyung.com/article/2025071876077,MICROSOFT
...,...,...,...,...,...
6155,"李 '세금 안 건드리겠다' 했는데…""집값 폭등 참사"" 경고한 진보 진영 [이슈+]",서울 잠실 롯데월드타워 전망대 서울스카이에서 바라본 도심 아파트 모습. 사진=김범준...,2025.06.16 12:09,https://www.hankyung.com/article/2025061674726,신한지주
6156,"李, 사법 리스크 넘기고 경제 행보 재개…내일 최태원 등 경제5단체장 간담회",사진=뉴스1\n\n 공직선거법 사건 파기환송심 재판 일정...,2025.05.07 17:17,https://www.hankyung.com/article/2025050776337,SK텔레콤
6157,"李, 정의선·구광모 한 달 새 두 번 만났다…재계 스킨십 강화",이재명 대통령이 7월 14일 서울 한남동 관저에서 정의선 현대차그룹 회장과 만찬 간...,2025.07.19 18:04,https://magazine.hankyung.com/business/article...,"[삼성전자, 현대차, 삼성전기]"
6158,"李·총수 간담회 후 첫 투자 소식…SK-아마존, 울산 AI 데이터센터에 수조원 투자",최태원 대한상공회의소 회장이 5월 8일 서울 중구 대한상공회의소에서 열린 대선 후보...,2025.06.16 10:21,https://magazine.hankyung.com/business/article...,SK텔레콤


## VectorDB 적용 전략
1. 아래 방식대로 VectorDB에 넣을 것임
1) Document에 넣을 값
 - content(내용)
2) meta에 추가할 예정
 - header (제목)
 - related_ticker (종목)
 - thema (주제)
 

### thema를 어떻게 정의하는 지 추후 더 발전시켜야 함

In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1. LLM 모델 세팅 (OpenAI)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# 2. 프롬프트 템플릿
prompt = ChatPromptTemplate.from_template("""
당신은 뛰어난 뉴스 기사 정리 전문가입니다
다음 뉴스 기사 내용을 읽고 주제를 분류하세요. 가능한 주제는 다음 네 가지입니다:
- 주가
- 시황
- 실적

위 단어 중 하나만 출력하세요.
만약에 위 3가지 단어로 정의할 수 없다면, 뉴스의 주제를 정의해주세요.
출력할 때는 "주가" 처럼 5문자 이내 단어 형태만 정의해주세요
                                          
[제목]
{header}

[내용]
{content}
""")

# 3. 출력 파서
parser = StrOutputParser()

# 4. 체인 구성
chain = prompt | llm | parser


In [21]:
def classify_thema(header: str, content: str) -> str:
    try:
        result = chain.invoke({"header": header, "content": content})
        return result.strip()
    except Exception as e:
        print("🔥 오류 발생:", e)
        return "오류"

In [22]:
for num, (header, content) in tqdm(enumerate(zip(news_df_filer.header.values, news_df_filer.content.values))):
    news_df_filer.loc[num,'thema'] = classify_thema(header=header, content=content)

873it [11:08,  1.31it/s]


KeyboardInterrupt: 

In [23]:
news_df_filer.thema.value_counts()

thema
시황      610
주가      139
실적      117
에너지       3
에어컨       1
기술        1
소비쿠폰      1
정치        1
Name: count, dtype: int64

In [ ]:

# 임베딩 모델
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

# 저장 경로
persist_directory = "../VectorDB/chroma_news_db"

# Chroma 벡터 저장소
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

In [30]:
def news_row_to_document(row):
    return Document(
        page_content=row["content"],
        metadata={
            "header": row["header"],
            "ticker": row["ticker"],
            "thema": row["thema"],
            "url" : row['url']
        }
    )

# 전체 뉴스 DataFrame → Document 리스트
documents = [news_row_to_document(row) for _, row in news_df_filer.iterrows()]

In [31]:
documents[0].page_content

"사진=뉴스1\n\n                “자동차산업의 위기가 곧 국가 제조업 전반의 위기로 이어질 수 있습니다. 정책적 뒷받침이 절실합니다.”자동차모빌리티산업연합회(KAIA)가 24일 서울 서초구 자동차회관 그랜저볼룸에서 '신정부에 바라는 자동차산업 정책과제'를 주제로 제42회 자동차모빌리티산업포럼을 열었다. 강남훈 회장은 이날 개회사에서 “자동차산업은 전후방 산업에 광범위한 영향을 미치는 '산업의 산업'으로, 약 150만 명에 이르는 직·간접 고용을 창출하며 우리 경제의 핵심 축 역할을 해왔다”며 “산업 생태계 전반의 불균형이 누적되고 있는 만큼 내수 활성화, 미래차 전환, 통상 대응, 인력 양성 등 전방위적인 정책 대응이 필요하다”고 강조했다.국내 자동차 산업이 미국의 수입차·부품 관세와 내수 경기 부진 등으로 어려움을 겪고 있는 가운데 신정부 출범에 맞춰 한목소리로 지원을 호소한 것이다. KAIA는 한국자동차모빌리티산업협회(KAMA), 한국자동차산업협동조합(KAICA), 한국자동차연구원(KATECH), 현대기아협력회, 한국지엠협신회, KG모빌리티파트너스 등 11개 단체로 구성된 연합체다.이날 첫 번째 주제발표에 나선 조철 산업연구원 선임연구위원은 미래차 지원 필요성을 역설했다. 그는 “자율주행과 커넥티드 기술의 진화가 가속화되면서 자동차의 소프트웨어화(SDV)와 인공지능 기술 역량이 기업 경쟁력을 좌우할 핵심 요소가 되고 있다”며 “이 과정에서 소프트웨어 중심의 생태계 조성과 부품업계의 기술 전환 대응력 제고가 정책적으로 뒷받침돼야 한다”고 강조했다. 미래차 부품산업 전환 촉진을 위한 특별법이 제정되었지만, 실질적 예산 반영이 미흡해 정책 실효성이 낮다는 지적이다.미국의 고율 관세에 정부가 적극적으로 대응해야 한다는 목소리도 컸다. 김영훈 한국자동차산업협동조합 실장은 “글로벌 공급망 재편과 미국의 보호무역 조치에 대응하기 위해, 현지화 투자 확대와 제도적 인프라 지원이 병행되어야 한다”고 강조하며 △북미 진출기업에 대한 금융·보증지원 확대 △KO

## chunking 필요
- 이유 : 기사 내용이 너무 많다

In [34]:
news_df_filer["content_length"] = news_df_filer["content"].str.len()
print(news_df_filer["content_length"].describe())

count      6160.000000
mean       1802.638312
std        5318.636776
min           0.000000
25%         695.000000
50%        1033.000000
75%        1541.000000
max      125274.000000
Name: content_length, dtype: float64


In [40]:
# ✅ 2. 텍스트 분할기 정의

def pick_splitter_by_length(text_len: int) -> RecursiveCharacterTextSplitter:
    """
    뉴스 본문의 길이에 따라 적절한 텍스트 분할기를 반환합니다.
    """
    if text_len <= 1200:
        # 짧은 기사 → 굳이 자르지 않고 1덩어리로 처리
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=0,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 10_000:
        # 중간 길이 → 일반적인 1,200자 기준으로 분할
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=150,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 50_000:
        # 긴 기사 → 덩어리를 좀 더 키움
        return RecursiveCharacterTextSplitter(
            chunk_size=1800,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    else:
        # 초장문 → 더 크게 자르되, 요약도 고려 (이건 후속 처리 필요)
        return RecursiveCharacterTextSplitter(
            chunk_size=2000,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    
def split_article(text: str):
    splitter = pick_splitter_by_length(len(text or ""))
    return splitter.split_text(text or "")


In [54]:
# 데이터 테이블에는 리스트 형태로 여러 종목이 들어가 있음
# 이 부분을 크로마DB에 넣을 때에는.. 문자열로만 넣어야해서, 문자열로 치환하는 작업
def serialize_ticker(ticker):
    if isinstance(ticker, (list, np.ndarray)):
        return ", ".join(map(str, ticker))
    elif pd.isna(ticker):
        return "None"
    return str(ticker)


In [55]:
# 2. 문서 리스트 생성 (chunk + metadata 포함)
def make_documents(df):
    docs = []

    for idx, row in df.iterrows():
        text = row["content"]
        splitter = pick_splitter_by_length(len(text))
        chunks = splitter.split_text(text)

        for i, chunk in enumerate(chunks):
            metadata = {
                "title": row["header"],
                "url": row["url"],
                "datetime": row["datetime"],
                "ticker": serialize_ticker(row.get("ticker", "None")),
                "thema": row.get("thema", "기타"),
                "chunk_idx": i,
                "original_idx": idx,
            }
            docs.append(Document(page_content=chunk, metadata=metadata))

    return docs


In [56]:
news_df_filer['content'] = news_df_filer['content'].fillna("")

In [57]:
# 3. 문서 분할 실행
documents = make_documents(news_df_filer)


In [58]:
# 4. OpenAI 임베딩 모델 로딩
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

# 5. Chroma DB 저장
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

In [ ]:
def make_doc_id(d: Document) -> str:
    """
    url + chunk_idx(없으면 0) 조합으로 안정적인 고유 id 생성
    """
    base = f"{d.metadata.get('url','')}_{d.metadata.get('chunk_idx', 0)}"
    return md5(base.encode("utf-8")).hexdigest()

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

# documents: 이미 만들어 둔 List[Document]
BATCH = 64  # 상황에 맞게 조절

for docs in tqdm(chunks(documents, BATCH), total=(len(documents) + BATCH - 1) // BATCH):
    ids = [make_doc_id(d) for d in docs]
    vectordb.add_documents(documents=docs, ids=ids)

100%|██████████| 202/202 [13:13<00:00,  3.93s/it] 


AttributeError: 'Chroma' object has no attribute 'persist'

## 크로마DB에 잘 있는 지 확인

In [76]:
# 전체 문서 수 확인
print("Number of documents in DB:", vectordb._collection.count())

Number of documents in DB: 0


In [64]:
results = vectordb.similarity_search("삼성전자 시황", k=3)
for doc in results:
    print(doc.metadata)
    print(doc.page_content[:300])

{'original_idx': 3962, 'url': 'https://www.hankyung.com/article/202507047931a', 'datetime': '2025.07.04 11:13', 'ticker': '에코프로, 에코프로비엠', 'title': '삼성전자, 글로벌 법리해석 불확실성·메모리 업황 관망에 보합', 'chunk_idx': 0}
삼성전자(005930)가 최근 뚜렷한 상승·하락 모멘텀 없는 가운데, 보합세 흐름을 이어가고 있다. 주요 원인으로는 글로벌 법률 리스크와 반도체 업황의 불확실성이 꼽힌다.먼저 미국 텍사스 연방법원의 특허 침해 소송 결과와 관련해, 일본 Maxell이 제기한 약 1.12억 달러 배상 판결은 항소를 예고하며 단기 투자심리의 부담 요인이 되고 있다. 이와 동시에 영국 고등법원은 ZTE 특허와 관련한 중간 라이선스 판결에서 삼성에 우호적 판단을 내려, 법적 승소 기대와 리스크 간 미묘한 균형이 형성된 상태다.또한 메모리 반도체 업황 회복 
{'ticker': '삼성전자', 'url': 'https://www.hankyung.com/article/202507159420g', 'chunk_idx': 2, 'original_idx': 5836, 'datetime': '2025.07.16 06:30', 'title': '하석진도 푹 빠졌다더니…요즘 삼성 가전 난리 난 이유'}
사진=삼성전자
{'original_idx': 3956, 'chunk_idx': 0, 'ticker': '삼성전자, SK하이닉스, 삼성SDI, 에코프로, 에코프로비엠', 'title': '삼성전자 주가 또 연중 최고치…나흘째 상승', 'url': 'https://www.hankyung.com/article/2025071865356', 'datetime': '2025.07.18 10:40'}
삼성전자 주가가 전날에 이어 18일 연중 최고가를 기록했다.이날 오전 10시28분 현재 삼성전자는 전날 대비 900원(1.35%) 오른6만7600원에 거래되고 있다.

## 추후 RAG 사용 시 Chunking된 docs가 아니라, 전체 DOCS가 있어야 함

In [65]:
def get_full_article_from_chroma(original_idx: int, vectordb) -> dict:
    """original_idx 기준으로 chunk들을 모아 원문 복원"""
    # 1. 해당 article의 모든 chunk 가져오기
    results = vectordb.similarity_search("", k=1000,  # 빈 쿼리로 전체 탐색
        filter={"original_idx": original_idx})

    if not results:
        return {"error": f"No chunks found for original_idx {original_idx}"}

    # 2. chunk 순서대로 정렬 (chunk_idx 기준)
    chunks_sorted = sorted(results, key=lambda d: d.metadata.get("chunk_idx", 0))

    # 3. 원문 내용 합치기
    full_content = "\n".join([chunk.page_content for chunk in chunks_sorted])

    # 4. 대표 metadata 하나 뽑아 저장
    meta = chunks_sorted[0].metadata
    return {
        "title": meta.get("title", ""),
        "url": meta.get("url", ""),
        "datetime": meta.get("datetime", ""),
        "ticker": meta.get("ticker", ""),
        "thema": meta.get("thema", ""),
        "content": full_content
    }

In [68]:
# 1. 사용자 쿼리로 VectorDB에서 chunk 검색
results = vectordb.similarity_search("삼성전자 시황", k=5)
print(results)

# 2. 검색된 chunk들의 original_idx 수집
original_indices = set([doc.metadata["original_idx"] for doc in results])

# 3. 각 original_idx에 대해 full article 복원
full_articles = [get_full_article_from_chroma(idx, vectordb) for idx in original_indices]

# 4. 이제 full_articles를 RAG의 context로 사용
context_texts = [article["content"] for article in full_articles]

[Document(id='9a3de35c2bcfdbd5c45b020e33a59d46', metadata={'ticker': '에코프로, 에코프로비엠', 'chunk_idx': 0, 'original_idx': 3962, 'url': 'https://www.hankyung.com/article/202507047931a', 'datetime': '2025.07.04 11:13', 'title': '삼성전자, 글로벌 법리해석 불확실성·메모리 업황 관망에 보합'}, page_content='삼성전자(005930)가 최근 뚜렷한 상승·하락 모멘텀 없는 가운데, 보합세 흐름을 이어가고 있다. 주요 원인으로는 글로벌 법률 리스크와 반도체 업황의 불확실성이 꼽힌다.먼저 미국 텍사스 연방법원의 특허 침해 소송 결과와 관련해, 일본 Maxell이 제기한 약 1.12억 달러 배상 판결은 항소를 예고하며 단기 투자심리의 부담 요인이 되고 있다. 이와 동시에 영국 고등법원은 ZTE 특허와 관련한 중간 라이선스 판결에서 삼성에 우호적 판단을 내려, 법적 승소 기대와 리스크 간 미묘한 균형이 형성된 상태다.또한 메모리 반도체 업황 회복 지연도 보합 흐름을 제한하는 요인이다. D램 가격은 여전히 안정권에 머무르고 있고 메모리 제외 반도체·가전 수요 회복도 더디다. 이로 인해 외국인 투자자 사이에서 관망세가 지속되고 있다.그러나 AI 관련 전략과 신제품 기대는 여전히 유효하다. 갤럭시와 가전 제품의 AI 기능 확대, 파운드리 부문 수주 가능성, 자사주 매입 계획 등은 중장기 주가 지지 요인으로 거론된다. 특히 외국인들은 메모리 실적 회복 기대에 따라 최근 매수에 나서기도 했다.종합하면, 삼성전자는 법적 리스크와 업황 회복 지연 속에서 단기 보합 흐름을 보이고 있으나, AI·반도체·M&A 등의 중장기 모멘텀으로 관망세가 유지되는 국면이다. 향후 관건은 ▶Maxell 항소 진행 ▶메모리 가격·수요 회복 ▶AI 신제품·파운드리 수주 성과 등으로 주가 방향을 결정할 것으로 보인다.현대모비스, 한화

In [67]:
context_texts

['[한경ESG] - ESG 핫 종목 -삼성전기\n글로벌 경기의 회복이 늦어지면서 전자부품 시장은 아직 힘을 쓰지 못하고 있다. 전자 부품주로 경기에 민감한 삼성전기는 침체된 시장 가운데 새로운 활로를 찾으면서 포트폴리오를 다시 짜고 있다. 스마트폰·PC 등 전통 IT 기기 중심의 포트폴리오에서 벗어나 AI·자율주행·서버 등 고성장 산업 중심으로 사업 방향을 틀었다. 신사업을 추진하면서도 친환경 경영 기조를 더욱 강화해 ESG 매력도 갖췄다는 평가다. 인류 미래 책임질 핵심 부품사로 삼성전기의 사업은 크게 세 부문으로 나뉜다. 적층세라믹콘덴서(MLCC)를 주력으로 한 컴포넌트, 패키지 기판인 FC-BGA 등이 포함된 패키지솔루션, 카메라모듈을 주축으로 하는 광학솔루션이다. 이 중 MLCC와 카메라모듈은 최근 들어 ‘전장용’ 비중이 급격히 늘고 있다. 2010년대까지만 해도 삼성전기의 MLCC는 거의 대부분 스마트폰과 TV에 들어갔다. 하지만 지금은 자동차 전장화가 급속도로 진행되면서 전기차, 첨단 운전자 보조 시스템(ADAS), 자율주행 시스템에 들어가는 고신뢰성 MLCC가 차세대 먹거리로 부상했다.특히 자동차용 고온·고전압 환경에 적합한 제품 수요가 증가하면서 삼성전기의 전장용 MLCC 매출 비중은 2022년 4%에서 2024년 20%를 넘어섰고, 2026년에는 30%까지 확대될 전망이다. 비슷한 변화는 카메라모듈에서도 감지된다.삼성전기는 스마트폰용 카메라모듈에서 출발했지만, 최근에는 전기차·로보택시의 ‘눈’ 역할을 하며 자율주행 시장으로 진입하고 있다. 특히 테슬라가 2025년부터 로보택시 상용화에 나서겠다고 밝힌 가운데 삼성전기의 멀티카메라 기술이 해당 시장에서 매력적인 옵션으로 부상 중이다. 인공지능(AI) 서버와 관련해서는 플립칩볼그리드어레이(FC-BGA)가 중심에 있다. 이는 고성능 칩과 메모리를 빠르고 안정적으로 연결하는 패키지 기판으로, 최근 AI 가속기용 칩 수요가 급증하면서 수혜를 받고 있다. 서버용 FC-BGA는 PC용보다 면적이 4배 크고

## RAG만 간단히 구현해보자!! CAG는 나중에!

- 추후 필요사항 : datetime을 고려해서 최신 기사만 추출해보자!
- 뉴스 기사 수집 자체 품질을 높여보자
- 삼성전자 기사 인데.. 한국경제 검색엔진을 취하다 보니.. 에코프로 쪽으로 분류되어있음..https://www.hankyung.com/article/202507047931a<br>
- ETF 구성 종목 내 명칭 그대로 검색해보니.. 제대로 수집을 못한 케이스도 존재

In [86]:
# 1. 벡터 DB와 LLM 준비
vectordb = Chroma(
    persist_directory="../VectorDB/chroma_news_db",
    embedding_function=embedding
)
llm = ChatOpenAI(model="gpt-4o", temperature=0)
parser = StrOutputParser()

# 2. 사용자 질문
query = "반도체 시황에서 최근에 가장 잘 나가는 회사가 어디야"

# 1. 사용자 쿼리로 VectorDB에서 chunk 검색
results = vectordb.similarity_search(query, k=5)
print(results)

# 2. 검색된 chunk들의 original_idx 수집
original_indices = set([doc.metadata["original_idx"] for doc in results])

# 3. 각 original_idx에 대해 full article 복원
full_articles = [get_full_article_from_chroma(idx, vectordb) for idx in original_indices]

# 4. 이제 full_articles를 RAG의 context로 사용
context_texts = [article["content"] for article in full_articles]

# 6. LLM 프롬프트 구성
prompt = ChatPromptTemplate.from_template("""
너는 금융 뉴스 분석 전문가야.

아래는 여러 금융 뉴스 기사들의 내용이야.  
- 이 문서들은 주로 주식시장 시황, 주가 변동, 기업 실적에 대한 기사야.
- 사용자 질문에 답할 때는 너가 아는 지식 말고, 항상 문서들을 참고해서 답해줘야해
- **가능하면 가장 최근 날짜의 기사**를 기준으로 답변해줘.
- 답변의 근거가 된 뉴스의 `제목(header)`, `링크(url)`, `날짜(datetime)`를 반드시 함께 명시해줘.
  이때 링크는 metadata의 url를 참고해서 답해줘

질문: {question}

뉴스 기사들:
{context}

답변:

""")

# 7. 프롬프트 → LLM 실행
rag_chain = prompt | llm | parser
response = rag_chain.invoke({
    "question": query,
    "context": "\n\n".join(context_texts)
})

# 8. 결과 출력
print(response)

[Document(id='9903ae3d77de43931acd7ec05f54cd1e', metadata={'datetime': '2025.07.03 14:53', 'ticker': 'SK하이닉스, 삼성전기', 'url': 'https://www.hankyung.com/article/202507035436a', 'chunk_idx': 0, 'original_idx': 5901, 'title': '한미반도체, HBM 열풍·관세 완화 기대에 주가 강세 지속'}, page_content='한미반도체(042700)가 글로벌 AI 반도체 수요 및 미국·중국 간 관세 불확실성 완화 기대를 중심으로 최근 주가가 강한 상승세를 이어가며 주목받고 있다.우선, HBM(고대역폭 메모리) 관련 장비 수요 확대가 실질적 모멘텀으로 작용하고 있다. 회사는 2023년 초부터 HBM용 ‘듀얼 TC본더’ 장비를 SK하이닉스와 공동 개발해 공급해왔으며, 이 기술력을 바탕으로 수익성이 크게 개선됐다. 실제로 주가는 지난 수년간 크게 상승했다.여기에 미국과 중국 간 관세 협상 진전 기대가 반도체 업종 전반에 긍정적으로 작용했다. 또한, SK하이닉스와의 TC본더 가격 논의 과정에서 일부 조정 장치가 마련되며, 시장에서는 기술력 우위뿐 아니라 공급 협상력에서도 우위를 점하고 있다는 평가가 나온다.한화오션, 두산, 삼성전기, 한미반도체, 크래프톤무료상담'), Document(id='a96c4a1288286a9f797c9021c051a0f5', metadata={'original_idx': 3691, 'url': 'https://www.hankyung.com/article/202506104391a', 'chunk_idx': 0, 'title': '반도체 관련주, 글로벌 수요 회복 기대감에 상승세', 'ticker': '한화에어로스페이스', 'datetime': '2025.06.10 10:23'}, page_content='최근 국내 증시에서 반도체 관련주가 상승세를 보이고 있다. 이는 글로벌 반도체 수요